# Transformer Triton kernel — Colab runner

Use a GPU runtime. This notebook invokes the same tests, manifest runner, and profiler as local validation. Colab results describe the assigned Colab GPU, not the curated RTX 5070 Ti run.

## 1. Clone the complete repository

Individual-file upload is insufficient because the kernel, dispatcher, manifests, and tools are separate modules. The next cell first tries anonymous access, which works once the submission repository is public. While it remains private, the cell securely prompts for a short-lived fine-grained GitHub token with read-only Contents access. The token is passed through a temporary `GIT_ASKPASS` helper, is not stored in the clone URL, and should be revoked after the run.

In [ ]:
import getpass
import os
import stat
import subprocess
import tempfile

repo_dir = '/content/tiktok-techjam-2026'
repo_url = 'https://github.com/lukeai-tan/tiktok-techjam-2026.git'
if not os.path.isdir(repo_dir):
    public_access = subprocess.run(
        ['git', 'ls-remote', repo_url, 'HEAD'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    ).returncode == 0
    if public_access:
        subprocess.run(['git', 'clone', repo_url, repo_dir], check=True)
    else:
        token = getpass.getpass('Fine-grained GitHub token (Contents: read-only): ')
        askpass_path = None
        clone_env = None
        try:
            with tempfile.NamedTemporaryFile('w', suffix='.sh', delete=False) as askpass:
                askpass.write("""#!/bin/sh
case "$1" in
  *Username*) printf '%s\n' 'x-access-token' ;;
  *) printf '%s\n' "$TIKTOK_TECHJAM_GITHUB_TOKEN" ;;
esac
""")
                askpass_path = askpass.name
            os.chmod(askpass_path, stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
            clone_env = os.environ.copy()
            clone_env.update({
                'GIT_ASKPASS': askpass_path,
                'GIT_TERMINAL_PROMPT': '0',
                'TIKTOK_TECHJAM_GITHUB_TOKEN': token,
            })
            subprocess.run(
                ['git', 'clone', repo_url, repo_dir],
                check=True,
                env=clone_env,
            )
        finally:
            token = None
            clone_env = None
            if askpass_path and os.path.exists(askpass_path):
                os.remove(askpass_path)
os.chdir(repo_dir)
print(os.getcwd())

## 2. Verify CUDA, PyTorch, and Triton

In [ ]:
import torch, triton
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('triton', triton.__version__)
print('gpu', torch.cuda.get_device_name(), 'capability', torch.cuda.get_device_capability())

## 3. Run the CPU/GPU contract suite

A compiler and Python headers may be required when Colab first builds Triton's driver shim.

In [ ]:
!python -m pip install -q pytest==9.1.1
!python -m pytest tests -q

## 4. Fast manifest smoke run

In [ ]:
!python benchmarks/run_matrix.py --device cuda --case long-causal-padding --dtype float32 --attention-backend auto --quick --accuracy-trials 3 --out results/colab-smoke.json

## 5. Full provisional matrix

This fails unless every requested case is PASS. It never converts compilation errors or OOM-only runs into success.

In [ ]:
!python benchmarks/run_matrix.py --device cuda --attention-backend auto --accuracy-trials 5 --out results/colab-matrix.json

## 6. Profiler proof

In [ ]:
!python benchmarks/profile_cases.py --case long-causal-padding --dtype float32 --attention-backend auto --steps 5 --out results/colab-profile.json

## 7. Download evidence

In [ ]:
from google.colab import files
files.download('results/colab-matrix.json')
files.download('results/colab-profile.json')